# Activation Oracle inference on saved Qwen3-8B activations

This notebook is an **inference-only** experiment for the saved Qwen3-8B activations.

It does **not** regenerate the target-model activations and does **not** train the Activation Oracle.

It follows the self-contained inference approach from the official Activation Oracle demo: the pretrained Qwen3-8B Activation Oracle adapter is attached to a Qwen3-8B base model, and the saved activation vectors are injected at special placeholder-token positions before generation.

Source: https://github.com/adamkarvonen/activation_oracles


## 1. Configuration

Upload your `dry_activations` folder to Kaggle as a dataset.

Expected files include:

```text
1_steered_L9_p.pt ... 1_steered_L27_o.pt
2_steered_L9_p.pt ... 2_steered_L27_o.pt
3_steered_L9_p.pt ... 3_steered_L27_o.pt
```

The notebook produces:

```text
oracle_outputs/
    input_1.json
    input_2.json
    input_3.json
```


In [1]:
'''
# Install only the packages needed by the self-contained demo.
# Kaggle normally already provides torch/transformers.
!pip install -q \
    "torch==2.7.1" \
    "transformers==4.55.2" \
    "peft==0.17.1" \
    "bitsandbytes==0.48.1"
'''
!pip install -q --upgrade \
    "bitsandbytes>=0.48.1" 
!pip install -q "torchao>0.16.0"



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.5 MB/s eta 0:00:00a 0:00:01


In [2]:
'''
torch: 2.10.0+cu128
transformers: 5.0.0
peft: 0.19.1
bitsandbytes: 0.50.2
CUDA: True
GPU: Tesla T4
'''

'\ntorch: 2.10.0+cu128\ntransformers: 5.0.0\npeft: 0.19.1\nbitsandbytes: 0.50.2\nCUDA: True\nGPU: Tesla T4\n'

In [3]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torch: 2.10.0+cu128
transformers: 5.0.0
peft: 0.19.1
bitsandbytes: 0.50.2
CUDA: True
GPU: Tesla T4


In [4]:
import os
import gc
import json
import re
from pathlib import Path
import contextlib
import sys

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [5]:

# ---- User configuration ----
ACTIVATIONS_DIR = "/kaggle/input/datasets/anarvaaa/activations-123"
OUTPUT_DIR = "/kaggle/working/oracle_outputs"

INPUT_IDS = [1]
LAYERS = [9, 18, 27]

# For A01-A08:
TASK_FOR_ACTIVATION = {
    "A01": "p", "A02": "p",
    "A03": "i", "A04": "i",
    "A05": "c", "A06": "c",
    "A07": "o", "A08": "o",
}

# A09-A12 are taken consistently from the p-file for the same input/layer.
COMMON_SOURCE_TASK = "p"

# Pretrained Activation Oracle checkpoint
# The source activation layer (9/18/27) is encoded in the oracle prompt.
# It is NOT the layer where the activation is injected into the oracle.
# The Activation Oracle injects supplied activations at its fixed read-in layer.
ORACLE_INJECTION_LAYER = 1

BASE_MODEL = "Qwen/Qwen3-8B"
ORACLE_ADAPTER = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

# Exactly the three questions we use for every activation.
QUESTIONS = {
    "Q1": "What kind of information does this activation appear to encode?",
    "Q2": "If this activation contains information relevant to a biomedical study, which of these is it most closely related to: population, intervention, comparator, outcome, or none of these?",
    "Q3": "Describe the specific biomedical concept, entity, or information represented by this activation, if one is identifiable. If no specific concept can be identified, say so rather than guessing.",
    "Q4": "Does this activation contain information that appears specifically related to the PICO extraction task? Answer yes, no, or uncertain, and briefly explain why."
}
'''
What task is this activation associated with?
Does this activation represent a population, intervention, comparator, or outcome concept?
What semantic information is most strongly represented here?
Which PICO element does this activation represent?
"What semantic concept or task does this activation represent?",
"What does this activation suggest the model is trying to do here?",
'''
# Oracle generation settings. These mirror the deterministic style of the demo.
MAX_NEW_TOKENS = 50
DO_SAMPLE = False

# Activation injection strength. The official hook defaults to 1.0.
STEERING_COEFFICIENT = 1.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configured.")


Configured.


## 2. Load the Qwen3-8B base + pretrained oracle adapter

The official demo loads `Qwen/Qwen3-8B` with 8-bit quantization and then uses the pretrained Qwen3-8B oracle adapter. The repository notes that its demo notebook is self-contained and does not import library code from the cloned repository.

We therefore do **not** clone `activation_oracles` in this Kaggle notebook.


In [6]:
import torch
import gc

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig

BASE_MODEL = "Qwen/Qwen3-8B"
ORACLE_ADAPTER = (
    "adamkarvonen/"
    "checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"
)

dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

print("Loading Qwen3-8B...")

# IMPORTANT:
# Keep device_map="auto" so the base model can be split/offloaded
# according to available GPU/CPU memory.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=dtype,
)

#quantization_config=quantization_config,

model.eval()

print("Base model loaded.")
print("Device map:")
print(model.hf_device_map)

# ------------------------------------------------------------
# Add the dummy adapter used by the official oracle setup
# ------------------------------------------------------------
dummy_config = LoraConfig()

model.add_adapter(
    dummy_config,
    adapter_name="default",
)

# ------------------------------------------------------------
# Load the pretrained Activation Oracle adapter.
#
# low_cpu_mem_usage=False lets us load it normally first.
# We then explicitly align only the LoRA modules to the device
# of their corresponding quantized base layer.
# ------------------------------------------------------------
print("Loading Activation Oracle adapter...")

model.load_adapter(
    ORACLE_ADAPTER,
    adapter_name="activation_oracle",
    is_trainable=False,
    low_cpu_mem_usage=False,
)

model.set_adapter("activation_oracle")
model.eval()

print("Oracle adapter loaded.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loading Qwen3-8B...


... (7 outputs left out)

adapter_model.safetensors:   0%|          | 0.00/698M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Qwen3ForCausalLM LOAD REPORT from: adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B
Key                                                          | Status  | 
-------------------------------------------------------------+---------+-
model.layers.{0...35}.self_attn.q_proj.lora_A.default.weight | MISSING | 
model.layers.{0...35}.self_attn.q_proj.lora_B.default.weight | MISSING | 
model.layers.{0...35}.self_attn.v_proj.lora_B.default.weight | MISSING | 
model.layers.{0...35}.self_attn.v_proj.lora_A.default.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Oracle adapter loaded.


In [7]:
import torch
import torch.nn as nn

def align_lora_adapters_to_base_layers(model):
    """
    With device_map='auto', the quantized base model can be split
    across GPU/CPU. PEFT adapters may initially remain on CPU.

    Move ONLY the LoRA adapter modules so that each adapter lives on
    the same device as its corresponding base layer.

    The large Qwen base model is not moved, so device_map='auto'
    and the memory-saving behavior are preserved.
    """
    moved = []

    for module_name, module in model.named_modules():
        if not hasattr(module, "base_layer"):
            continue

        base_layer = module.base_layer

        # Find the device of this quantized base layer.
        try:
            base_device = base_layer.weight.device
        except Exception:
            params = list(base_layer.parameters())
            if not params:
                continue
            base_device = params[0].device

        # Standard LoRA A/B modules.
        for attr_name in ("lora_A", "lora_B"):
            container = getattr(module, attr_name, None)
            if container is None:
                continue

            for adapter_name, adapter_module in container.items():
                old_device = next(
                    adapter_module.parameters()
                ).device

                if old_device != base_device:
                    adapter_module.to(base_device)
                    moved.append(
                        (module_name, attr_name, adapter_name,
                         str(old_device), str(base_device))
                    )

        # Support LoRA embedding containers if present.
        for attr_name in ("lora_embedding_A", "lora_embedding_B"):
            container = getattr(module, attr_name, None)
            if container is None:
                continue

            for adapter_name, value in container.items():
                if hasattr(value, "device") and value.device != base_device:
                    try:
                        container[adapter_name].data = value.data.to(base_device)
                        moved.append(
                            (module_name, attr_name, adapter_name,
                             str(value.device), str(base_device))
                        )
                    except Exception:
                        pass

    return moved


moved_lora = align_lora_adapters_to_base_layers(model)

print(f"LoRA tensors/modules moved: {len(moved_lora)}")

if moved_lora:
    print("\nFirst few alignment changes:")
    for item in moved_lora[:10]:
        print(item)
else:
    print("No LoRA device corrections were needed.")


LoRA tensors/modules moved: 504

First few alignment changes:
('model.layers.0.self_attn.q_proj', 'lora_A', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.q_proj', 'lora_B', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.k_proj', 'lora_A', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.k_proj', 'lora_B', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.v_proj', 'lora_A', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.v_proj', 'lora_B', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.o_proj', 'lora_A', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.self_attn.o_proj', 'lora_B', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.mlp.gate_proj', 'lora_A', 'activation_oracle', 'cpu', 'cuda:0')
('model.layers.0.mlp.gate_proj', 'lora_B', 'activation_oracle', 'cpu', 'cuda:0')


### Device-map-auto note

The base Qwen3-8B remains under `device_map="auto"`. Only the small LoRA adapter modules are moved to the same device as their corresponding quantized base layer. The saved activations themselves remain on CPU until the hook executes, where they are moved to the current layer's `resid.device`.


In [8]:
# Minimal pieces of the official Activation Oracle inference mechanism.
#
# IMPORTANT LAYER DISTINCTION
# ---------------------------
# `source_layer` = the layer in the ORIGINAL Qwen3-8B model from which
#                  the saved activation came (e.g. 9, 18, 27).
#
# `ORACLE_INJECTION_LAYER` = the layer in the ORACLE Qwen3-8B model where
#                            the saved activation is injected.
#
# These are NOT the same layer.
#
# The source layer is communicated through:
#     "Layer: {source_layer}\n..."
#
# The activation itself is injected into the oracle at layer 1.

SPECIAL_TOKEN = " ?"


def get_introspection_prefix(source_layer: int, num_positions: int) -> str:
    return (
        f"Layer: {source_layer}\n"
        + SPECIAL_TOKEN * num_positions
        + " \n"
    )


def get_hf_submodule(model, source_layer: int):
    """
    Return the FIXED oracle injection layer.

    `source_layer` is intentionally NOT used to select the oracle block.
    It only identifies where the supplied activation came from.
    """
    injection_layer = ORACLE_INJECTION_LAYER

    candidates = [
        ("model.model.layers", lambda m: m.model.layers),
        ("model.base_model.model.layers", lambda m: m.base_model.model.layers),
        ("model.base_model.layers", lambda m: m.base_model.layers),
        ("model.layers", lambda m: m.layers),
    ]

    for name, getter in candidates:
        try:
            obj = getter(model)
            if hasattr(obj, "__getitem__"):
                candidate_layer = obj[injection_layer]

                print(
                    f">>> Source activation layer: {source_layer}"
                )
                print(
                    f">>> Oracle injection layer: {injection_layer}"
                )
                print(f">>> Using layer path: {name}")
                print(f">>> Number of oracle layers: {len(obj)}")
                print(
                    f">>> Oracle layer {injection_layer}: "
                    f"{type(candidate_layer)}"
                )

                return candidate_layer

        except (AttributeError, TypeError, IndexError):
            continue

    raise AttributeError(
        "Could not locate the oracle Qwen3 transformer layers."
    )


def find_pattern_in_tokens(
    token_ids,
    special_token_str,
    num_positions,
    tokenizer,
):
    special_token_id = tokenizer.encode(
        special_token_str,
        add_special_tokens=False,
    )

    if len(special_token_id) != 1:
        raise ValueError(
            f"Expected {special_token_str!r} to be one token, "
            f"got {special_token_id}"
        )

    sid = special_token_id[0]

    positions = []
    for i, tid in enumerate(token_ids):
        if tid == sid:
            positions.append(i)
            if len(positions) == num_positions:
                break

    if len(positions) != num_positions:
        raise ValueError(
            f"Expected {num_positions} placeholder positions, "
            f"found {len(positions)}"
        )

    if positions[-1] - positions[0] != num_positions - 1:
        raise ValueError(
            f"Placeholder positions are not consecutive: {positions}"
        )

    return positions


def add_hook(module, hook):
    handle = module.register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


In [9]:
# Verify the ACTIVE oracle adapter and fixed injection layer.

print("Source layers used in the experiment:", LAYERS)
print("Oracle injection layer:", ORACLE_INJECTION_LAYER)

oracle_layer = get_hf_submodule(model, ORACLE_INJECTION_LAYER)

devices = set()
for p in oracle_layer.parameters():
    devices.add(str(p.device))

print(f"Oracle injection layer {ORACLE_INJECTION_LAYER} devices: {devices}")

cpu_lora = []
cuda_lora = []

for name, param in model.named_parameters():
    if "lora_" in name.lower():
        if param.device.type == "cpu":
            cpu_lora.append((name, tuple(param.shape)))
        elif param.device.type == "cuda":
            cuda_lora.append((name, tuple(param.shape)))

print("LoRA parameters on CUDA:", len(cuda_lora))
print("LoRA parameters on CPU :", len(cpu_lora))

if cpu_lora:
    print("\nSome LoRA parameters are still on CPU:")
    for item in cpu_lora[:10]:
        print(item)


Source layers used in the experiment: [9, 18, 27]
Oracle injection layer: 1
>>> Source activation layer: 1
>>> Oracle injection layer: 1
>>> Using layer path: model.model.layers
>>> Number of oracle layers: 36
>>> Oracle layer 1: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>
Oracle injection layer 1 devices: {'cuda:0'}
LoRA parameters on CUDA: 648
LoRA parameters on CPU : 0


In [10]:

import contextlib

@contextlib.contextmanager
def temporary_hook(module, hook):
    handle = module.register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()

def make_activation_hook(vectors, positions, steering_coefficient=1.0):
    """
    Inject the saved activation only during the initial prompt/prefill pass.

    During autoregressive generation, Qwen uses KV cache and subsequent
    forward passes can have sequence length L=1. The original placeholder
    positions no longer exist in those decode steps, so the hook must leave
    those later outputs untouched.
    """
    vectors = [v.detach() for v in vectors]

    if len(vectors) != len(positions):
        raise ValueError("vectors and positions must have the same batch length")

    normed = [F.normalize(v, dim=-1).detach() for v in vectors]

    def hook_fn(module, _input, output):
        if isinstance(output, tuple):
            resid, *rest = output
            output_is_tuple = True
        else:
            resid = output
            rest = []
            output_is_tuple = False

        B, L, D = resid.shape

        if B != len(normed):
            raise ValueError(
                f"Batch mismatch: model batch={B}, vectors={len(normed)}"
            )

        # IMPORTANT:
        # On the initial prefill, L is the full oracle-prompt length and
        # the placeholder positions are present.
        #
        # During generation with KV cache, L can become 1. The saved
        # placeholder positions do not exist in that step, so do NOT
        # inject again.
        max_required_position = max(
            max(pos) for pos in positions if len(pos) > 0
        )

        if max_required_position >= L:
            return output

        for b in range(B):
            pos = torch.tensor(
                positions[b],
                dtype=torch.long,
                device=resid.device,
            )

            if pos.numel() == 0:
                raise ValueError("No injection positions")

            if int(pos.max()) >= L:
                # Defensive guard for any shorter/cached decode pass.
                continue

            orig = resid[b, pos, :]
            orig_norm = orig.norm(dim=-1, keepdim=True)

            injected = (
                normed[b].to(resid.device)
                * orig_norm
                * steering_coefficient
            ).to(resid.dtype)

            resid[b, pos, :] = orig + injected.detach()

        return (resid, *rest) if output_is_tuple else resid

    return hook_fn

## 3. Helpers for loading the saved `.pt` activations

The saved structure is expected to be:

```python
{
    "activations": {
        "A01": tensor(...),
        ...
    },
    "metadata": {...}
}
```

A01–A08 use the task-specific files:

- A01/A02 → `p`
- A03/A04 → `i`
- A05/A06 → `c`
- A07/A08 → `o`

A09–A12 use the `p` file consistently for each input/layer.


In [11]:

def find_activation_file(input_id: int, layer: int, task: str) -> Path:
    path = Path(ACTIVATIONS_DIR) / f"{input_id}_steered_L{layer}_{task}.pt"
    if not path.exists():
        raise FileNotFoundError(f"Missing activation file: {path}")
    return path

def load_activation_dict(path: Path):
    obj = torch.load(path, map_location="cpu")
    if not isinstance(obj, dict):
        raise TypeError(f"{path} did not contain a dictionary")
    if "activations" not in obj:
        raise KeyError(f"{path} has no 'activations' key")
    return obj

def canonicalize_activation_tensor(t):
    """
    Convert:
      [D]       -> [1, D]
      [1, D]    -> [1, D]
      [1, S, D] -> [S, D]
      [S, D]    -> [S, D]

    The oracle expects one activation vector per placeholder token.
    """
    if not torch.is_tensor(t):
        raise TypeError(f"Expected Tensor, got {type(t)}")

    t = t.detach().float().cpu()

    if t.ndim == 1:
        return t.unsqueeze(0)

    if t.ndim == 2:
        return t

    if t.ndim == 3:
        if t.shape[0] != 1:
            raise ValueError(f"Unexpected batch dimension: {tuple(t.shape)}")
        return t[0]

    raise ValueError(f"Unexpected activation shape: {tuple(t.shape)}")

def inspect_one_file():
    p = find_activation_file(INPUT_IDS[0], LAYERS[0], "p")
    obj = load_activation_dict(p)
    print("File:", p)
    print("Top-level keys:", list(obj.keys()))
    print("Activation keys:", list(obj["activations"].keys()))
    for k, v in obj["activations"].items():
        print(f"  {k}: {tuple(v.shape)}")

inspect_one_file()


File: /kaggle/input/datasets/anarvaaa/activations-123/1_steered_L9_p.pt
Top-level keys: ['activations', 'metadata']
Activation keys: ['A01', 'A02', 'A09', 'A10', 'A11', 'A12']
  A01: (4096,)
  A02: (3, 4096)
  A09: (4096,)
  A10: (1, 4096)
  A11: (476, 4096)
  A12: (177, 4096)


## 4. Oracle probing function

**Important:** the official demo calls `apply_chat_template(..., return_tensors=None)` and then constructs the tensor itself. This avoids a `BatchEncoding` object being passed to `torch.ones_like()`.

For each activation:

1. Create the official-style introspection prefix with the same number of `?` placeholder positions as there are activation vectors.
2. Add the oracle question.
3. Apply the Qwen chat template.
4. Find the exact placeholder-token positions.
5. Inject the supplied saved activation vectors at those positions.
6. Generate the oracle answer.

A01–A08 therefore receive **only their saved token/span activation**, not the complete original sequence.


In [12]:
@torch.no_grad()
def ask_oracle_with_activation(
    activation,
    layer: int,
    question: str,
    max_new_tokens: int = 50,
):
    # ------------------------------------------------------------
    # 1. Prepare saved activation
    # ------------------------------------------------------------
    vectors = canonicalize_activation_tensor(activation)

    num_positions = vectors.shape[0]

    if vectors.shape[1] != model.config.hidden_size:
        raise ValueError(
            f"Hidden size mismatch: activation has {vectors.shape[1]}, "
            f"oracle model expects {model.config.hidden_size}"
        )

    # ------------------------------------------------------------
    # 2. Build the Activation Oracle prompt
    # ------------------------------------------------------------
    # `layer` is the SOURCE layer of the saved activation.
    # It is included in the oracle prompt as metadata.
    prompt = get_introspection_prefix(layer, num_positions) + question

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    # ------------------------------------------------------------
    # 3. Tokenize explicitly as a BatchEncoding
    # ------------------------------------------------------------
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        padding=False,
        enable_thinking=False,
    )

    # With device_map="auto", do NOT use model.device blindly.
    # Put the input on the device hosting the input embeddings.
    input_device = model.get_input_embeddings().weight.device

    input_ids = tokenized["input_ids"].to(input_device)

    if "attention_mask" in tokenized:
        attention_mask = tokenized["attention_mask"].to(input_device)
    else:
        attention_mask = torch.ones_like(input_ids)

    # ------------------------------------------------------------
    # 4. Find placeholder positions
    # ------------------------------------------------------------
    token_list = input_ids[0].tolist()

    positions = find_pattern_in_tokens(
        token_list,
        SPECIAL_TOKEN,
        num_positions,
        tokenizer,
    )

    # ------------------------------------------------------------
    # 5. Find the FIXED oracle injection transformer block
    #    (source layer != oracle injection layer)
    # ------------------------------------------------------------
    injection_submodule = get_hf_submodule(model, layer)

    # IMPORTANT:
    # Keep vectors on CPU here. The hook moves them to resid.device
    # at the exact layer being executed. This is necessary when
    # device_map="auto" places different layers on different devices.
    hook = make_activation_hook(
        [vectors],
        [positions],
        steering_coefficient=STEERING_COEFFICIENT,
    )

    # ------------------------------------------------------------
    # 6. Generate oracle explanation
    # ------------------------------------------------------------
    with temporary_hook(injection_submodule, hook):
        with open(os.devnull, "w") as devnull:
            with contextlib.redirect_stderr(devnull):
                output_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_new_tokens,
                    do_sample=DO_SAMPLE,
                    )

    # ------------------------------------------------------------
    # 7. Decode ONLY newly generated tokens
    # ------------------------------------------------------------
    generated = output_ids[:, input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated[0],
        skip_special_tokens=True,
    ).strip()

    return {
        "question": question,
        "answer": answer,
        "num_positions": int(num_positions),
        "activation_shape": list(activation.shape),
        "oracle_prompt": prompt,
    }


## 5. Build the A01–A12 source map

For each input and layer:

```text
A01/A02 ← p file
A03/A04 ← i file
A05/A06 ← c file
A07/A08 ← o file
A09/A10/A11/A12 ← p file
```

A13/A14 are intentionally not included in this experiment.


In [13]:

def build_source_map(input_id: int, layer: int):
    sources = {}

    # A01-A08
    for aid, task in TASK_FOR_ACTIVATION.items():
        path = find_activation_file(input_id, layer, task)
        obj = load_activation_dict(path)

        if aid not in obj["activations"]:
            raise KeyError(
                f"{aid} not found in {path}. Found: {list(obj['activations'].keys())}"
            )

        sources[aid] = {
            "path": str(path),
            "task_file": task,
            "activation": obj["activations"][aid],
            "metadata": obj.get("metadata", {}),
        }

    # A09-A12
    common_path = find_activation_file(input_id, layer, COMMON_SOURCE_TASK)
    common_obj = load_activation_dict(common_path)

    for aid in ["A09", "A10", "A11", "A12"]:
        if aid not in common_obj["activations"]:
            raise KeyError(
                f"{aid} not found in {common_path}. "
                f"Found: {list(common_obj['activations'].keys())}"
            )

        sources[aid] = {
            "path": str(common_path),
            "task_file": COMMON_SOURCE_TASK,
            "activation": common_obj["activations"][aid],
            "metadata": common_obj.get("metadata", {}),
        }

    return sources


## 6. Run the three-input × three-layer × A01–A12 experiment

This is the main inference cell.

For each input:

- L9
- L18
- L27

and each:

- A01–A12
- Q1–Q3

the oracle generates one explanation.

That is:

**3 inputs × 3 layers × 12 activations × 3 questions = 324 oracle generations.**



In [14]:

def run_all():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_results = {}

    for input_id in INPUT_IDS:
        print(f"\n{'='*80}")
        print(f"INPUT {input_id}")
        print(f"{'='*80}")

        input_result = {
            "input_id": input_id,
            "model": BASE_MODEL,
            "oracle_adapter": ORACLE_ADAPTER,
            "layers": {},
        }

        for layer in LAYERS:
            print(f"\n--- Layer {layer} ---")
            sources = build_source_map(input_id, layer)

            layer_result = {}

            for aid in [f"A{i:02d}" for i in range(1, 13)]:
                src = sources[aid]
                act = src["activation"]

                activation_result = {
                    "source_file": src["path"],
                    "source_task": src["task_file"],
                    "original_shape": list(act.shape),
                    "oracle_probes": {},
                }

                print(f"{aid}: {Path(src['path']).name} | shape={tuple(act.shape)}")

                for qid, question in QUESTIONS.items():
                    result = ask_oracle_with_activation(
                        activation=act,
                        layer=layer,
                        question=question,
                        max_new_tokens=MAX_NEW_TOKENS,
                    )
                    activation_result["oracle_probes"][qid] = result

                    print(f"  {qid}: {result['answer'][:180]}")

                    # Encourage memory release between individual generations.
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                layer_result[aid] = activation_result

            input_result["layers"][f"L{layer}"] = layer_result

        out_path = Path(OUTPUT_DIR) / f"input_{input_id}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(input_result, f, indent=2, ensure_ascii=False)

        print(f"\nSaved: {out_path}")
        all_results[input_id] = input_result

    return all_results

results = run_all()



INPUT 1

--- Layer 9 ---


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A01: 1_steered_L9_p.pt | shape=(4096,)
>>> Source activation layer: 9
>>> Oracle injection layer: 1
>>> Using layer path: model.model.layers
>>> Number of oracle layers: 36
>>> Oracle layer 1: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>
  Q1: This activation encodes information about the specific proteins and their interactions within the cell.
>>> Source activation layer: 9
>>> Oracle injection layer: 1
>>> Using layer path: model.model.layers
>>> Number of oracle layers: 36
>>> Oracle layer 1: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>
  Q2: The activation is most closely related to the 'outcome' in a biomedical study.
>>> Source activation layer: 9
>>> Oracle injection layer: 1
>>> Using layer path: model.model.layers
>>> Number of oracle layers: 36
>>> Oracle layer 1: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3DecoderLayer'>
  Q3: The activation corresponds to the concept of a blood sample being collected for analysis.
>

## 7. Verify the saved JSON files

Each JSON keeps the source `.pt` filename, activation shape, question, and oracle answer.

Example structure:

```text
input_1.json
└── layers
    ├── L9
    │   ├── A01
    │   ├── A02
    │   ├── ...
    │   └── A12
    ├── L18
    └── L27
```


In [15]:

for input_id in INPUT_IDS:
    path = Path(OUTPUT_DIR) / f"input_{input_id}.json"
    print(path, "exists:", path.exists())
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(
            "  layers:", list(data["layers"].keys()),
            "| L9 activations:", list(data["layers"]["L9"].keys())
        )
        print()


/kaggle/working/oracle_outputs/input_1.json exists: True
  layers: ['L9', 'L18', 'L27'] | L9 activations: ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12']

